<a href="https://colab.research.google.com/github/giannismantzaris-cmd/DAMA60/blob/main/Mantzaris_Topic_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import networkx as nx
from typing import List, Tuple, Dict

# ---------- Graph definition (nodes 1..9) ----------
EDGES = [(1,2),(1,3),(2,3),(2,4),(3,4),(3,5),(4,6),(5,6),(6,7),(7,8),(7,9)]



def build_W_column_stochastic_from_nx(G: nx.Graph) -> Tuple[List[int], np.ndarray, Dict[int, int]]:
    """
    Constructs the column-stochastic transition probability matrix W of a graph.

    Parameters
    ----------
    G : nx.Graph
        An undirected NetworkX graph. Nodes are assumed to be integers.

    Returns
    -------
    nodes : List[int]
        Sorted list of node labels.
    W : np.ndarray
        (n x n) column-stochastic transition matrix where
        W[j, i] = probability of moving from node i to node j.
    deg : Dict[int, int]
        Dictionary mapping each node to its degree.
    """
    nodes = sorted(G.nodes())

    # Compute the degree of each node.
    deg = {u: G.degree(u) for u in nodes}

    # Get the number of nodes and initialize the transition matrix with zeros.
    n = len(nodes)
    W = np.zeros((n, n), dtype=float)

    # Create a mapping from node label to matrix index.
    node_to_idx = {u: i for i, u in enumerate(nodes)}

    # Fill the matrix column by column.
    for u in nodes:
        # Get the column index of the current node.
        i = node_to_idx[u]

        # If the node has neighbors, assign equal probability to each neighbor.
        if deg[u] > 0:
            for v in G.neighbors(u):
                # Get the row index of the neighbor.
                j = node_to_idx[v]

                # Set the transition probability from u to v.
                W[j, i] = 1.0 / deg[u]

    return nodes, W, deg

def rwr_iterations(W: np.ndarray, nodes: List[int], seed_node: int, c: float = 0.85, T: int = 4) -> List[np.ndarray]:
    """
    Performs Random Walk with Restart (RWR) iterations.

    Parameters
    ----------
    W : np.ndarray
        (n x n) column-stochastic transition matrix.
    nodes : List[int]
        Ordered list of nodes corresponding to the indexing of W.
    seed_node : int
        Query node from which the random walk starts.
    c : float, optional
        Continuation probability (default = 0.85).
    T : int, optional
        Number of RWR iterations (default = 4).

    Returns
    -------
    R : List[np.ndarray]
        List containing the ranking vectors:
        [r^(0), r^(1), ..., r^(T)].
    """
    # Get the number of nodes in the graph.
    n = len(nodes)

    # Initialize the restart / preference vector with zeros.
    e = np.zeros(n, dtype=float)

    # Find the position of the seed node in the ordered node list.
    seed_idx = nodes.index(seed_node)

    # Set probability 1 at the seed node.
    e[seed_idx] = 1.0

    # Initialize the first ranking vector r^(0) as the restart vector.
    r0 = e.copy()

    # Create the list of ranking vectors starting with r^(0).
    R = [r0]

    # Perform T RWR iterations.
    for _ in range(T):
        # Compute the next ranking vector using the RWR update formula.
        r_next = c * W @ R[-1] + (1 - c) * e

        # Store the new ranking vector.
        R.append(r_next)


    return R

def similarity_ranking(r: np.ndarray, nodes: List[int], seed_node: int) -> List[Tuple[int, float]]:
    """
    Computes similarity ranking of nodes with respect to the query node.

    Parameters
    ----------
    r : np.ndarray
        Final ranking vector (n x 1).
    nodes : List[int]
        Ordered list of nodes corresponding to vector r.
    seed_node : int
        Query node that must be excluded from the ranking.

    Returns
    -------
    List[Tuple[int, float]]
        List of (node, score) pairs sorted in descending order of similarity,
        excluding the seed_node.
    """
    # Create an empty list to store (node, score) pairs.
    pairs = []

    # Iterate over all nodes and their corresponding scores.
    for i, node in enumerate(nodes):
        # Exclude the seed node from the ranking.
        if node != seed_node:
            # Append the node and its score.
            pairs.append((node, r[i]))

    # Sort the pairs in descending order based on the score.
    pairs.sort(key=lambda x: x[1], reverse=True)



    return pairs

if __name__ == "__main__":
    G = nx.Graph()
    G.add_edges_from(EDGES)

    nodes, W, deg = build_W_column_stochastic_from_nx(G)
    seed = 3
    c = 0.85
    T = 4

    R = rwr_iterations(W, nodes, seed_node=seed, c=c, T=T)

    np.set_printoptions(precision=6, suppress=True)
    print("Nodes:", nodes)
    print("Degrees:", {u: deg[u] for u in nodes})
    print("\nTransition matrix W (destination rows, source columns):\n", W)

    for t in range(T+1):
        print(f"\nr^({t}) =\n", R[t].reshape(-1))

    rank = similarity_ranking(R[T], nodes, seed_node=seed)
    print("\nSimilarity ranking based on r^(4) (excluding node 3):")
    for node, score in rank:
        print(f"Node {node}: {score:.6f}")




Nodes: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Degrees: {1: 2, 2: 3, 3: 4, 4: 3, 5: 2, 6: 3, 7: 3, 8: 1, 9: 1}

Transition matrix W (destination rows, source columns):
 [[0.       0.333333 0.25     0.       0.       0.       0.       0.
  0.      ]
 [0.5      0.       0.25     0.333333 0.       0.       0.       0.
  0.      ]
 [0.5      0.333333 0.       0.333333 0.5      0.       0.       0.
  0.      ]
 [0.       0.333333 0.25     0.       0.       0.333333 0.       0.
  0.      ]
 [0.       0.       0.25     0.       0.       0.333333 0.       0.
  0.      ]
 [0.       0.       0.       0.333333 0.5      0.       0.333333 0.
  0.      ]
 [0.       0.       0.       0.       0.       0.333333 0.       1.
  1.      ]
 [0.       0.       0.       0.       0.       0.       0.333333 0.
  0.      ]
 [0.       0.       0.       0.       0.       0.       0.333333 0.
  0.      ]]

r^(0) =
 [0. 0. 1. 0. 0. 0. 0. 0. 0.]

r^(1) =
 [0.2125 0.2125 0.15   0.2125 0.2125 0.     0.     0.     0.    ]

r^(2) =